### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="santander_customer_satisfaction",
    dataset_year="2016",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/santander-customer-satisfaction",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c santander-customer-satisfaction -f train.csv && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/santander_customer_satisfaction && mv train.csv local-data-warehouse/santander_customer_satisfaction/
""",
    # References
    academic_reference_bibtex=r"""@misc{Jimenez2016SantanderCustomerSatisfaction,
  author = {Soraya Jimenez and Will Cukierski},
  title  = {Santander Customer Satisfaction},
  year   = {2016},
  howpublished = {\url{https://kaggle.com/competitions/santander-customer-satisfaction}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Jimenez2016SantanderCustomerSatisfaction",
    license="Kaggle Competition Rules",
    data_tags=["IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized, so feature meanings are unknown.
- We remove duplicated columns and zero-variance columns.
- We add a zero-count feature (number of zeros per row). #1 Kaggle experts also applied a lot of other features and feature selection but each experts used something different (https://www.kaggle.com/competitions/santander-customer-satisfaction/writeups/1-leustagos-3rd-place-solution).
- We drop duplicates as their origin is unclear and they could create data leaks when splitting the training data.
- We replace -999999 in var3 with with nan.
- We keep the data otherwise as it. Several columns might be categorical and contain encoded nan values but we are not able to determine this due to anonymization.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="TARGET",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="TARGET",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "train.csv")
print("Loaded data shape:", df.shape)

# remove duplicated columns
df = df.loc[:, ~df.T.duplicated()]
# remove zero-variance columns
df = df.loc[:, df.nunique(dropna=False) > 1]
# add zero-count feature
numeric_cols = df.drop(columns=["ID", "TARGET"])
df["zero_count"] = (numeric_cols == 0).sum(axis=1)
df = df.drop(columns=["ID"])
df[task_mold.target_column_name] = df[task_mold.target_column_name].astype("category")

# Drop duplicates w/o target col
df = df.drop_duplicates(subset=df.columns.difference([task_mold.target_column_name]))

# Replace -999999 with np.nan
df["var3"] = df["var3"].replace(-999999, np.nan)

df = df.reset_index(drop=True)

Loaded data shape: (76020, 371)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 71,080
Columns: 308
Use sampling: False (sample size: 71,080)


Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['var38', 'saldo_medio_var5_ult3', 'saldo_var30', 'saldo_var42', 'saldo_medio_var5_ult1', 'saldo_medio_var5_hace2', 'saldo_var5', 'imp_op_var39_comer_ult3', 'imp_op_var41_comer_ult3', 'imp_op_var39_ult1']
Rows remaining as candidates after top-10 filter: 5,180 (of 71,080)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,imp_op_var40_ult1,imp_op_var41_comer_ult1,imp_op_var41_comer_ult3,imp_op_var41_efect_ult1,imp_op_var41_efect_ult3,imp_op_var41_ult1,imp_op_var39_efect_ult1,imp_op_var39_efect_ult3,imp_op_var39_ult1,imp_sal_var16_ult1,ind_var1_0,ind_var1,ind_var5_0,ind_var5,ind_var6_0,ind_var6,ind_var8_0,ind_var8,ind_var12_0,ind_var12,ind_var13_0,ind_var13_corto_0,ind_var13_corto,ind_var13_largo_0,ind_var13_largo,ind_var13_medio_0,ind_var13,ind_var14_0,ind_var14,ind_var17_0,ind_var17,ind_var18_0,ind_var19,ind_var20_0,ind_var20,ind_var24_0,ind_var24,ind_var25_cte,ind_var26_0,ind_var26_cte,ind_var25_0,ind_var30_0,ind_var30,ind_var31_0,ind_var31,ind_var32_cte,ind_var32_0,ind_var33_0,ind_var33,ind_var34_0,ind_var37_cte,ind_var37_0,ind_var39_0,ind_var40_0,ind_var40,ind_var41_0,ind_var44_0,ind_var44,num_var1_0,num_var1,num_var4,num_var5_0,num_var5,num_var6_0,num_var6,num_var8_0,num_var8,num_var12_0,num_var12,num_var13_0,num_var13_corto_0,num_var13_corto,num_var13_largo_0,num_var13_largo,num_var13_medio_0,num_var13,num_var14_0,num_var14,num_var17_0,num_var17,num_var18_0,num_var20_0,num_var20,num_var24_0,num_var24,num_var26_0,num_var25_0,num_op_var40_hace2,num_op_var40_hace3,num_op_var40_ult1,num_op_var40_ult3,num_op_var41_hace2,num_op_var41_hace3,num_op_var41_ult1,num_op_var41_ult3,num_op_var39_hace2,num_op_var39_hace3,num_op_var39_ult1,num_op_var39_ult3,num_var30_0,num_var30,num_var31_0,num_var31,num_var32_0,num_var33_0,num_var33,num_var34_0,num_var35,num_var37_med_ult2,num_var37_0,num_var39_0,num_var40_0,num_var40,num_var41_0,num_var42_0,num_var42,num_var44_0,num_var44,saldo_var1,saldo_var5,saldo_var6,saldo_var8,saldo_var12,saldo_var13_corto,saldo_var13_largo,saldo_var13_medio,saldo_var13,saldo_var14,saldo_var17,saldo_var18,saldo_var20,saldo_var24,saldo_var26,saldo_var25,saldo_var30,saldo_var31,saldo_var32,saldo_var33,saldo_var34,saldo_var37,saldo_var40,saldo_var42,saldo_var44,var36,delta_imp_amort_var18_1y3,delta_imp_amort_var34_1y3,delta_imp_aport_var13_1y3,delta_imp_aport_var17_1y3,delta_imp_aport_var33_1y3,delta_imp_compra_var44_1y3,delta_imp_reemb_var13_1y3,delta_imp_reemb_var17_1y3,delta_imp_reemb_var33_1y3,delta_imp_trasp_var17_in_1y3,delta_imp_trasp_var17_out_1y3,delta_imp_trasp_var33_in_1y3,delta_imp_trasp_var33_out_1y3,delta_imp_venta_var44_1y3,delta_num_aport_var13_1y3,delta_num_aport_var17_1y3,delta_num_aport_var33_1y3,delta_num_compra_var44_1y3,delta_num_venta_var44_1y3,imp_amort_var18_ult1,imp_amort_var34_ult1,imp_aport_var13_hace3,imp_aport_var13_ult1,imp_aport_var17_hace3,imp_aport_var17_ult1,imp_aport_var33_hace3,imp_aport_var33_ult1,imp_var7_emit_ult1,imp_var7_recib_ult1,imp_compra_var44_hace3,imp_compra_var44_ult1,imp_reemb_var13_ult1,imp_reemb_var17_hace3,imp_reemb_var17_ult1,imp_reemb_var33_ult1,imp_var43_emit_ult1,imp_trans_var37_ult1,imp_trasp_var17_in_hace3,imp_trasp_var17_in_ult1,imp_trasp_var17_out_ult1,imp_trasp_var33_in_hace3,imp_trasp_var33_in_ult1,imp_trasp_var33_out_ult1,imp_venta_var44_hace3,imp_venta_var44_ult1,ind_var7_emit_ult1,ind_var7_recib_ult1,ind_var10_ult1,ind_var10cte_ult1,ind_var9_cte_ult1,ind_var9_ult1,ind_var43_emit_ult1,ind_var43_recib_ult1,var21,num_aport_var13_hace3,num_aport_var13_ult1,num_aport_var17_hace3,num_aport_var17_ult1,num_aport_var33_hace3,num_aport_var33_ult1,num_var7_emit_ult1,num_var7_recib_ult1,num_compra_var44_hace3,num_compra_var44_ult1,num_ent_var16_ult1,num_var22_hace2,num_var22_hace3,num_var22_ult1,num_var22_ult3,num_med_var22_ult3,num_med_var45_ult3,num_meses_var5_ult3,num_meses_var8_ult3,num_meses_var12_ult3,num_meses_var13_corto_ult3,num_meses_var13_largo_ult3,num_meses_var13_medio_ult3,num_meses_var17_ult3,num_meses_var29_ult3,num_meses_var33_ult3,num_meses_var39_vig_ult3,num_meses_var44_ult3,num_op_var39_comer_ult1,num_op_var39_comer_ult3,num_op_var40_comer_ult1,num_op_var40_comer_ult3,num_op_var40_efect_u

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,TARGET,category,0.0,0.00,2.0,"0, 1"
1,var3,float64,107.0,0.15,207.0,"2.0, 8.0, 9.0, 3.0, 1.0, 13.0, 7.0, 4.0, 12.0, 6.0"
2,imp_ent_var16_ult1,float64,0.0,0.00,596.0,"0.0, 300.0, 150.0, 600.0, 900.0, 3.0, 450.0, 60.0, 1500.0, 90.0"
3,imp_op_var39_comer_ult1,float64,0.0,0.00,7551.0,"0.0, 30.0, 60.0, 15.0, 90.0, 4.5, 45.0, 150.0, 120.0, 2.67"
4,imp_op_var39_comer_ult3,float64,0.0,0.00,9099.0,"0.0, 30.0, 60.0, 4.5, 15.0, 90.0, 45.0, 150.0, 2.67, 120.0"
5,imp_op_var40_comer_ult1,float64,0.0,0.00,293.0,"0.0, 396.0, 30.0, 180.0, 175.47, 168.0, 2189.04, 1222.62, 1430.94, 488.43"
6,imp_op_var40_comer_ult3,float64,0.0,0.00,346.0,"0.0, 30.0, 4.5, 3520.59, 109.5, 1430.31, 678.48, 499.2, 2189.04, 1222.62"
7,imp_op_var40_efect_ult1,float64,0.0,0.00,23.0,"0.0, 900.0, 1800.0, 60.0, 600.0, 450.0, 300.0, 120.0, 270.0, 210.0"
8,imp_op_var40_efect_ult3,float64,0.0,0.00,29.0,"0.0, 900.0, 1800.0, 120.0, 960.0, 330.0, 60.0, 600.0, 450.0, 300.0"
9,imp_op_var40_ult1,float64,0.0,0.00,224.0,"0.0, 600.0, 450.0, 900.0, 1800.0, 567.0, 270.0, 828.18, 199.53, 17.94"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
var3,70973.0,2.767489e+00,9.776127e+00,0.00,2.380000e+02
var15,71080.0,3.355101e+01,1.310395e+01,5.00,1.050000e+02
imp_ent_var16_ult1,71080.0,9.219967e+01,1.669762e+03,0.00,2.100000e+05
imp_op_var39_comer_ult1,71080.0,7.739224e+01,3.503540e+02,0.00,1.288803e+04
imp_op_var39_comer_ult3,71080.0,1.278368e+02,5.639895e+02,0.00,2.102481e+04
imp_op_var40_comer_ult1,71080.0,3.806486e+00,9.633366e+01,0.00,8.237820e+03
imp_op_var40_comer_ult3,71080.0,6.922545e+00,1.589799e+02,0.00,1.107357e+04
imp_op_var40_efect_ult1,71080.0,4.416458e-01,3.165032e+01,0.00,6.600000e+03
imp_op_var40_efect_ult3,71080.0,6.067829e-01,3.776073e+01,0.00,6.600000e+03
imp_op_var40_ult1,71080.0,3.380383e+00,9.851941e+01,0.00,8.237820e+03


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column rank                    
TARGET 1        0  68368  96.18
       2        1   2712   3.82

In [8]:
# Target Distribution
target_df

,count,pct
TARGET,,
0,68368,96.18
1,2712,3.82


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to santander_customer_satisfaction/019d5dc6-86ea-7525-81f2-e50b404706ea


019d5dc6-86ea-7525-81f2-e50b404706ea
db64be8fb612418697885b5fad51f3a2256f6a19e8d80aecf88fa6fb6066d9c2
